# 00 · Guía del Laboratorio 2

**Pronóstico de caudal a 48 h con Adapformer** (Luo et al., *Neural Networks* 2026).

Implementación propia de Adapformer en PyTorch, adaptada a pronosticar el caudal específico (mm/h) de las próximas 48 h
a partir de 336 h de 12 variables. El informe completo está en `INFORME.md`.

| Notebook | Contenido | Requisito del enunciado |
|---|---|---|
| `01_datos_y_eda` | Formato de los datos, particiones, por qué `log1p`, persistencia, dónde está el error | Carga de datos y particiones · preprocesamiento |
| `02_adapformer_paso_a_paso` | Cada componente del paper con tensores reales; por qué el SimBlock sólo aprende por la pérdida auxiliar | Implementación del método del paper |
| `03_entrenamiento` | Pérdida, bucle de entrenamiento, modelos base | Entrenamiento y validación · modelo base |
| `04_resultados_y_ablacion` | Comparación con los modelos base, por horizonte/régimen/cuenca, ablaciones, canales elegidos | Métricas · ablación |
| `05_errores_test_y_entrega` | Análisis de errores, limitaciones, `submission.csv`, reporte en test | Predicciones de test · análisis de errores |

## Estructura

```
Entrega_Lab2/
├── INFORME.md                informe del laboratorio
├── README.md                 instrucciones breves
├── configs/                  un YAML por experimento (modelo + entrenamiento + resultado medido)
│   ├── 01_final/             final_cd.yaml → MODELO FINAL
│   ├── 02_base/              LSTM base · Adapformer k = 4 (top-k adaptativo del paper)
│   ├── 03_ablacion/          k = 1, sin ACE, sin pérdida auxiliar
│   ├── 04_mejoras_descartadas/   pérdida híbrida, token de cuenca, EMA, embedding MLP, r = 64
│   └── 05_exploracion/       primera exploración (una semilla)
├── src/
│   ├── datos.py              lectura de los H5, log1p + z-score, caché float16, carga a GPU
│   ├── modelos.py            Adapformer (RevIN, SimBlock, ACE, ACF) + LSTM base
│   ├── entrenar.py           bucle de entrenamiento (línea de comandos o YAML)
│   ├── metricas.py           MSE, MAE, NSE/KGE por cuenca, por horizonte y por régimen
│   ├── resultados.py         tablas, ensambles, prueba de Wilcoxon
│   ├── hacer_configs.py      regenera configs/
│   └── fase*.sh, _comun.sh   experimentos en el orden en que se corrieron
├── notebooks/                00–05, ejecutados
├── figuras/                  figuras del informe
├── referencias_estudio1/     métricas de la versión inicial (k = 4, una semilla), usadas como comparación
└── outputs/
    ├── submission.csv        predicciones de test
    ├── stats.json            media y desviación por canal (train)
    └── <corrida>/            resumen.json y log.csv de cada corrida;
                              best.pt y predicciones sólo en final_cd_s* y lstm_s*
```

`cache/` (≈ 2.6 GB) no se incluye: lo crea `python src/datos.py`.

**Dónde va el dataset** (no se incluye en la entrega). La carpeta con `train.h5`, `test.h5` y `metadata.json` se busca en
este orden: (1) la ruta de la variable de entorno `LAB2_DATA_DIR`; (2) `Entrega_Lab2/dataset/`; (3) `dataset/` al lado de
`Entrega_Lab2/`. `test_targets.csv` (métricas de test) y `leer_datos.py` (notebook 01) son opcionales. Si falta algún
archivo obligatorio, `python src/datos.py` se detiene y muestra las ubicaciones válidas.

## Cómo reproducir todo

Entorno: Python 3.12, PyTorch ≥ 2.4 con CUDA, h5py, numpy, pandas, scipy, matplotlib, pyyaml, jupyter. Primero
coloque el dataset (ver arriba). Desde la carpeta `Entrega_Lab2/`:

```bash
python src/datos.py                 # 1) caché preprocesada (una sola vez)
bash src/fase1_mejoras.sh           # 2) exploración, 1 semilla (pérdidas, token de cuenca)
bash src/fase1b_semillas.sh         # 3) 3 semillas: k = 4, pérdida híbrida λ = 1/3, token de cuenca
bash src/fase1c_ema.sh              # 4) 3 semillas: EMA
bash src/fase2_final.sh             # 5) LSTM base, modelo final k = 12 y ablación (3 semillas)
bash src/fase3_cd.sh                # 6) ablación de ACE sobre el modelo final
bash src/fase4_embedding.sh         # 7) embedding MLP y r = 64
python -m jupyter nbconvert --to notebook --execute --inplace notebooks/0*.ipynb   # 8) notebooks
```

Los scripts saltan las corridas que ya existen. Una corrida individual se lanza con su YAML:

```bash
python src/entrenar.py --config configs/01_final/final_cd.yaml --seed 42      # → outputs/final_cd_s42
python src/entrenar.py --config configs/01_final/final_cd.yaml --seed 7 --set r=64 --run prueba_r64
```

Comprobado: el YAML del modelo final reproduce exactamente la corrida original (val MSE 0.011200).

## Tiempos medidos en esta laptop

In [1]:
import json, re
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
filas = []
for p in sorted((ROOT / "outputs").glob("*/resumen.json")):
    if p.parent.name.startswith("demo"): continue
    r = json.loads(p.read_text())
    n = r["run"]
    fase = ("1 · exploración" if n.startswith("af_") else "1b/1c · semillas" if n.startswith("p1_")
            else "2 · LSTM" if n.startswith("lstm") else "3 · ablación de ACE en k=12" if n.startswith("cd_noace")
            else "4 · embedding MLP y r=64" if n.startswith("cd_") else "2 · final k=12 + ablación")
    filas.append({"fase": fase, "corrida": r["run"], "modelo": r["cfg"]["model"]["name"], "épocas": r["epochs_run"],
                  "mejor época": r["best_epoch"], "minutos": r["minutos"]})
t = pd.DataFrame(filas)
display(t.groupby("fase").agg(corridas=("corrida", "count"), minutos_total=("minutos", "sum"), minutos_por_corrida=("minutos", "mean")).round(1))
print(f"Total de entrenamiento: {t.minutos.sum():.0f} min en {len(t)} corridas")

,corridas,minutos_total,minutos_por_corrida
fase,,,
1 · exploración,4,9.5,2.4
1b/1c · semillas,15,34.3,2.3
2 · LSTM,3,8.8,2.9
2 · final k=12 + ablación,12,21.3,1.8
3 · ablación de ACE en k=12,3,2.8,0.9
4 · embedding MLP y r=64,6,6.7,1.1


Total de entrenamiento: 83 min en 43 corridas


> **Nota de hardware:** RTX 4060 Laptop (8 GB). Con la laptop en modo de energía *Silent* las épocas tardan ~30 s en
> lugar de ~5 s (la GPU queda esperando a la CPU); los tiempos de la tabla son en modo *Performance*.

## Resumen

1. **Versión inicial:** Adapformer con top-k adaptativo (k = 4), pérdida en espacio log y anclaje al último caudal.
   Con 3 semillas obtiene val MSE 0.01205 ± 0.00012.
2. **Cinco mejoras propuestas no funcionaron** (pérdida híbrida log + mm/h, token de cuenca, EMA de pesos, embedding
   MLP y r = 64). Los notebooks 04 y 05 explican por qué: la respuesta de la cuenca a la lluvia es muy incierta.
3. **Lo que sí funcionó:** usar **k = 12** en el ACF (−6 % de MSE en las 3 semillas, mejor NSE y KGE) y **ensamblar
   3 semillas**. Modelo entregado (`outputs/submission.csv`): val MSE 0.01117 (persistencia 0.01901) y test MSE
   0.01086 (−23 % frente a la persistencia).
4. **Ablación:** ACE es el componente con efecto más claro; la selección adaptativa top-k no ayuda con 12 canales.
5. **Frente al LSTM:** empate práctico (algo peor en validación, algo mejor en test).